In [1]:
import os
import time
import json
import tracemalloc
import numpy as np

# PySCF & IPIE Imports
from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.utils.mpi import MPIHandler

# IPIE Analysis Imports
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable

# =============================================================================
# 0. CONFIGURATION
# =============================================================================
N_ATOMS = 10
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048
TEST_SEED = 999
NUM_BLOCKS = 50

# =============================================================================
# 1. SETUP MPI & PYSCF
# =============================================================================
comm = MPIHandler()
rank = comm.rank

chk_file = f"scf_h{N_ATOMS}.chk"
ham_file = f"ham_h{N_ATOMS}.h5"
wfn_file = f"wfn_h{N_ATOMS}.h5"

if rank == 0:
    print(f">>> STARTING CLASSICAL AFQMC BASELINE FOR: {SYSTEM_NAME}")
    
    # Generate geometry (H10 chain)
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.chkfile = chk_file
    
    # Run SCF if integrals don't exist
    if not os.path.exists(chk_file) or not os.path.exists(wfn_file):
        print(f"[Main] Running PySCF and generating integrals...")
        CURRENT_HF_ENERGY = mf.kernel()
        gen_ipie_input_from_pyscf_chk(chk_file, hamil_file=ham_file, wfn_file=wfn_file, verbose=0, chol_cut=1e-5)
    else:
        print(f"[Main] Reading HF Energy from: {chk_file}")
        CURRENT_HF_ENERGY = lib.chkfile.load(chk_file, 'scf/e_tot')
        if CURRENT_HF_ENERGY is None:
            CURRENT_HF_ENERGY = mf.kernel()
            gen_ipie_input_from_pyscf_chk(chk_file, hamil_file=ham_file, wfn_file=wfn_file, verbose=0, chol_cut=1e-5)
            
    print(f"[Main] Current PySCF HF Energy: {CURRENT_HF_ENERGY:.6f} Ha")
else:
    CURRENT_HF_ENERGY = None

comm.comm.Barrier()

# =============================================================================
# 2. INITIALIZE AFQMC
# =============================================================================
if rank == 0:
    print("\n" + "="*50)
    print(f">>> Building AFQMC Object (Seed: {TEST_SEED})")
    print("="*50)

afqmc = AFQMC.build_from_hdf5(
    num_elec=(N_ATOMS//2, N_ATOMS//2),
    ham_file=ham_file,
    wfn_file=wfn_file,
    num_blocks=NUM_BLOCKS,
    num_steps_per_block=1,
    num_walkers=WALKERS,
    seed=TEST_SEED,
    verbose=0
)

afqmc.mpi_handler = comm
if comm.size > 1:
    local_walkers = WALKERS // comm.size
    if rank < (WALKERS % comm.size): local_walkers += 1
    afqmc.nwalkers = local_walkers

# =============================================================================
# 3. RUN WITH PROFILING
# =============================================================================
# Start memory allocation tracker
if rank == 0:
    tracemalloc.start()
    start_time = time.perf_counter()

# Run the classical physics engine
afqmc.run()

# Stop tracking
if rank == 0:
    end_time = time.perf_counter()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

# =============================================================================
# 4. DATA EXTRACTION, ANALYSIS & SAVING
# =============================================================================
if rank == 0:
    print("\n" + "="*50)
    print(">>> PERFORMING AUTOCORRELATION ANALYSIS")
    print("="*50)
    
    final_energy = None
    final_error = None
    
    try:
        # Extract the observable data from the estimators file
        estimator_filename = afqmc.estimators.filename
        qmc_data = extract_observable(estimator_filename, "energy")
        
        # Discard the first block to account for equilibration
        y_energy = qmc_data["ETotal"][1:]
        
        # Perform autocorrelation reblocking to get true statistical error
        df_ac = reblock_by_autocorr(y_energy, verbose=0)
        
        # Extract the relevant values from the dataframe
        final_energy = float(df_ac["ETotal_ac"].iloc[0])
        final_error = float(df_ac["ETotal_error_ac"].iloc[0])
        
        print(f"  Final Reblocked Energy: {final_energy:.6f} Ha")
        print(f"  Final Reblocked Error:  {final_error:.6f} Ha")
        
    except Exception as e:
        print(f"  [Warning] Autocorrelation analysis failed: {e}")
        print("  Falling back to raw standard error of the mean.")
        if 'y_energy' in locals() and len(y_energy) > 0:
            final_energy = float(np.mean(y_energy))
            final_error = float(np.std(y_energy) / np.sqrt(len(y_energy)))
            print(f"  Fallback Raw Mean Energy: {final_energy:.6f} ± {final_error:.6f} Ha")

    # Get Hamiltonian Dimensions
    try:
        n_basis = afqmc.system.nbasis
        n_chol = afqmc.hamiltonian.nchol
    except AttributeError:
        # Fallback if ipie attributes vary by version
        n_basis = N_ATOMS  # sto-6g H has 1 basis function per atom
        n_chol = getattr(afqmc.hamiltonian, 'nchol', N_ATOMS * 3) # rough estimate

    # Time calculations
    total_time = end_time - start_time
    time_per_block = total_time / NUM_BLOCKS
    
    # Memory conversions (Bytes to MB)
    peak_mem_mb = peak_mem / (1024 ** 2)
    
    # Theoretical FLOPs for Cholesky local energy evaluation
    flops_per_step = 8 * WALKERS * (n_basis ** 2) * n_chol
    total_flops = flops_per_step * NUM_BLOCKS
    teraflops_per_sec = (total_flops / total_time) / 1e12

    metrics = {
        "system": SYSTEM_NAME,
        "n_atoms": N_ATOMS,
        "n_basis": n_basis,
        "n_cholesky_vectors": n_chol,
        "total_walkers": WALKERS,
        "num_blocks": NUM_BLOCKS,
        "results": {
            "final_energy_ha": final_energy,
            "final_error_ha": final_error
        },
        "timing": {
            "total_wall_time_sec": round(total_time, 4),
            "time_per_block_sec": round(time_per_block, 4)
        },
        "memory": {
            "peak_rss_mb": round(peak_mem_mb, 2)
        },
        "compute": {
            "est_flops_per_step": flops_per_step,
            "est_total_flops": total_flops,
            "est_tflops_throughput": round(teraflops_per_sec, 6)
        }
    }

    print("\n" + "="*50)
    print(">>> BASELINE SCALING METRICS SUMMARY")
    print("="*50)
    print(f"  Final Energy:         {final_energy} ± {final_error} Ha")
    print(f"  Total Wall Time:      {total_time:.4f} sec")
    print(f"  Time Per Block:       {time_per_block:.4f} sec")
    print(f"  Peak Memory (Rank 0): {peak_mem_mb:.2f} MB")
    print(f"  System Dimensions:    Basis={n_basis}, Cholesky={n_chol}")
    print(f"  Est. Total FLOPs:     {total_flops:.2e}")
    print("="*50)

    # Save to disk
    save_path = f"baseline_metrics_{SYSTEM_NAME}.json"
    with open(save_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Metrics saved to {save_path}")

>>> STARTING CLASSICAL AFQMC BASELINE FOR: H10
[Main] Reading HF Energy from: scf_h10.chk
[Main] Current PySCF HF Energy: -5.096510 Ha

>>> Building AFQMC Object (Seed: 999)
# random seed is 999
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -1.0437649726627393e+04  2.0480000000000000e+03 -5.0965086555797816e+00 -1.7754760831862484e+01  1.2658252176282703e+01
                1   2.0616780949217637e+03  2.0480000000000000e+03 -2.6630942934985069e+00 -1.0509588414829544e+04  2.0616780949217637e+03 -5.0975894057934203e+00 -1.7754758987919356e+01  1.2657169582125935e+01
                2   2.0617445384773637e+03  2.0480000000000000e+03 -2.6759482301032143e+00 -1.0512091286110026e+04  2.0617445384773637e+03 -5.0986390844878384e+00 -1.775475